# Fine-Tuning a Large Language Model for Layperson Medical Question Answering
AI in Healthcare | MSAI Spr '25 | Chelsea Ramos

## Objectives
1. ***Explore and sample the MedQuAD dataset***
2. Generate medical multiple-choice questions (MCQs) using a MedAlpaca
3. Fine-tune TinyLLaMA with the generated MCQs
4. Evaluate model accuracy on held-out MCQs
5. Compare against other models (baseline and zero-shot)

---

# 1. Explore and Sample MedQuAD

In [1]:
import pandas as pd

## Load & Clean Data

- Dataset: MedQuAD 
    - 47,457 layperson medical question-answer pairs

In [2]:
df = pd.read_parquet("hf://datasets/lavita/MedQuAD/data/train-00000-of-00001-e36383d177026d53.parquet")
print('Shape:', df.shape)
df.head()

Shape: (47441, 13)


,document_id,document_source,document_url,category,umls_cui,umls_semantic_types,umls_semantic_group,synonyms,question_id,question_focus,question_type,question,answer
0,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-1,keratoderma with woolly hair,information,What is (are) keratoderma with woolly hair ?,Keratoderma with woolly hair is a group of rel...
1,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-2,keratoderma with woolly hair,frequency,How many people are affected by keratoderma wi...,Keratoderma with woolly hair is rare; its prev...
2,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-3,keratoderma with woolly hair,genetic changes,What are the genetic changes related to kerato...,"Mutations in the JUP, DSP, DSC2, and KANK2 gen..."
3,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-4,keratoderma with woolly hair,inheritance,Is keratoderma with woolly hair inherited ?,Most cases of keratoderma with woolly hair hav...
4,0000559,GHR,https://ghr.nlm.nih.gov/condition/keratoderma-...,None,C0343073,T047,Disorders,KWWH,0000559-5,keratoderma with woolly hair,treatment,What are the treatments for keratoderma with w...,These resources address the diagnosis or manag...


### Drop Missing Answers

In [3]:
df.isna().sum()

document_id                5
document_source            0
document_url               0
category               15431
umls_cui               16024
umls_semantic_types    16066
umls_semantic_group    16024
synonyms               22772
question_id                0
question_focus            14
question_type              0
question                   0
answer                 31034
dtype: int64

In [4]:
df.dropna(subset=['answer'], inplace=True)
print('Shape:', df.shape)

Shape: (16407, 13)


### Drop Duplicate QAs

In [5]:
print('Duplicates:', df.duplicated(subset=['question', 'answer']).sum())
df.drop_duplicates(subset=['question', 'answer'], inplace=True)
print('New Shape:', df.shape)

Duplicates: 48
New Shape: (16359, 13)


### Clean Up Questions and Answers

In [6]:
# Get max word count of question and answer
def get_max_length(df, column):
    return df[column].str.split().str.len().max()

max_question_length = get_max_length(df, 'question')
max_answer_length = get_max_length(df, 'answer')
print('Max question length:', max_question_length)
print('Max answer length:', max_answer_length)

Max question length: 27
Max answer length: 4281


In [7]:
# Remove excess whitespace from answer
df["answer"] = df["answer"].str.replace(r"[^\S\r\n]+", " ", regex=True).str.strip()
# Remove space before question mark
df["question"] = df["question"].str.replace(r"\s+\?", "?", regex=True)
# Remove question from answer
df["answer"] = df["answer"].str.replace(r"^.*?\?\s*", "", regex=True).str.strip()
# Keep answers that are not too long
df = df[df['answer'].str.split().str.len() <= 150]
max_answer_length = get_max_length(df, 'answer')
print('New max answer length:', max_answer_length)
print('Shape:', df.shape)

New max answer length: 150
Shape: (8861, 13)


### Drop Small question_type Groups

In [8]:
df['question_type'].value_counts()

question_type
information        2331
treatment          1910
inheritance        1200
frequency          1118
causes              445
outlook             338
symptoms            309
genetic changes     292
exams and tests     284
research            260
susceptibility      169
prevention           92
considerations       86
complications        21
stages                5
support groups        1
Name: count, dtype: int64

In [9]:
small_groups = ['stages', 'complications', 'support groups', 'prevention', 'considerations']
df = df.loc[~(df['question_type'].isin(small_groups))]
print('Shape:', df.shape)

Shape: (8656, 13)


## Sample and Save QAs

In [10]:
def balanced_group_sample(df, group_cols, n_per_group):
    # Group by specified cols
    grouped = df.groupby(group_cols)
    # Sample n rows (or all, if size is less than n) per group
    balanced_sample = grouped.apply(
        lambda x: x.sample(n=min(n_per_group, len(x)), random_state=42),
        include_groups=False
    ).reset_index()
    return balanced_sample

sampled_df = balanced_group_sample(df, group_cols=["question_type"], n_per_group=300)

# Check shape and distribution
print('Shape:', sampled_df.shape)
sampled_df['question_type'].value_counts()


Shape: (3105, 14)


question_type
causes             300
frequency          300
information        300
inheritance        300
outlook            300
symptoms           300
treatment          300
genetic changes    292
exams and tests    284
research           260
susceptibility     169
Name: count, dtype: int64

In [11]:
# Save sampled data to CSV
sampled_df.to_csv('../data/medquad_sampled.csv', index=False)
# Save QA only to jsonl
sampled_df = sampled_df[['question', 'answer']]
sampled_df.to_json('../data/medquad_sampled.jsonl', orient='records', lines=True)
sampled_df

,question,answer
0,What causes Thalassemia?,"There are two main types of thalassemia, alpha..."
1,What causes Holes in the Heart?,Mothers of children who are born with atrial s...
2,What causes Bronchiolitis obliterans organizin...,"BOOP may be caused by a variety of factors, in..."
3,What causes Simple Kidney Cysts?,The cause of simple kidney cysts is not fully ...
4,What causes Tietze syndrome?,The exact underlying cause of Tietze syndrome ...
...,...,...
3100,What are the treatments for hereditary hemorrh...,These resources address the diagnosis or manag...
3101,What are the treatments for Multiple System At...,There is no cure for multiple system atrophy w...
3102,What are the treatments for Familial hemiplegi...,Treatment of hemiplegic migraine varies depend...
3103,What are the treatments for Herpes zoster oticus?,Treatment for herpes zoster oticus typically i...
